# House Price Prediction (Regression)
Predict house prices based on structural and location features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded')

## 1. Load Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
try:
    df = pd.read_csv(url)
    if 'medv' in df.columns:
        df.rename(columns={'medv': 'SalePrice'}, inplace=True)
    print('Boston Housing dataset loaded')
except Exception as e:
    print('URL failed, using synthetic data:', e)
    np.random.seed(42)
    n = 1500
    df = pd.DataFrame({
        'LotArea':      np.random.randint(1500, 20000, n),
        'OverallQual':  np.random.randint(1, 11, n),
        'OverallCond':  np.random.randint(1, 10, n),
        'YearBuilt':    np.random.randint(1900, 2010, n),
        'TotalBsmtSF':  np.abs(np.random.normal(1000, 300, n)).astype(int).astype(float),
        'GrLivArea':    np.abs(np.random.normal(1500, 500, n)).astype(int),
        'FullBath':     np.random.randint(1, 4, n),
        'BedroomAbvGr': np.random.randint(1, 6, n),
        'GarageCars':   np.random.randint(0, 4, n),
        'Neighborhood': np.random.choice(['NAmes', 'CollgCr', 'OldTown', 'Edwards'], n),
        'SalePrice':    np.abs(np.random.normal(180000, 60000, n))
    })
    df.loc[np.random.choice(n, 60, replace=False), 'TotalBsmtSF'] = np.nan
    df.loc[np.random.choice(n, 5, replace=False), 'SalePrice'] = 900000
df.head()

## 2. Data Types of All Columns

In [ ]:
print('Shape:', df.shape)
print()
print(df.dtypes)

## 3. Descriptive Statistics

In [ ]:
df.describe().T

## 4. Identify and Handle Missing Values

In [ ]:
print('Missing values per column:')
mv = df.isnull().sum()
print(mv[mv > 0])

for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print('Missing after handling:', df.isnull().sum().sum())

## 5. Identify and Handle Duplicates

In [ ]:
print(f'Duplicates: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

## 6. Identify and Handle Outliers (IQR Method)

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    before = len(df)
    df = df[(df[col] >= lower) & (df[col] <= upper)]
    removed = before - len(df)
    if removed:
        print(f'  {col}: removed {removed} outliers')
df.reset_index(drop=True, inplace=True)
print(f'Shape after outlier removal: {df.shape}')

## 7. Visualizations & Insights

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['SalePrice'], bins=40, color='seagreen', edgecolor='white')
axes[0].set_title('Sale Price Distribution')
axes[0].set_xlabel('Price ($)')

num_df = df.select_dtypes(include='number')
corr_with_price = num_df.corr()['SalePrice'].drop('SalePrice').sort_values()
corr_with_price.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Correlation with SalePrice')
axes[1].axvline(0, color='black', linewidth=0.8)

sns.heatmap(num_df.corr(), ax=axes[2], annot=True, fmt='.2f', cmap='RdYlGn', linewidths=0.4)
axes[2].set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

print('Insights:')
print('1. SalePrice is right-skewed; most houses are in the mid-range.')
print('2. OverallQual and GrLivArea have the strongest positive correlation with price.')
print('3. Older YearBuilt negatively correlates with price.')

## 8. Feature Selection, Encoding & Scaling

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

num_df = df.select_dtypes(include='number')
corr_matrix = num_df.corr()
selected = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > 0.1].index.tolist()
selected = [c for c in selected if c != 'SalePrice']
print('Selected features:', selected)

X = df[selected]
y = df['SalePrice']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 9. Model Building (Linear Regression, Ridge, Random Forest)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'MAE':  round(mean_absolute_error(y_test, preds), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 2),
        'R2':   round(r2_score(y_test, preds), 3)
    }
    print(f'{name} trained')

## 10. Model Performance Comparison

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['MAE', 'RMSE', 'R2']
colors = ['#4C72B0', '#DD8452', '#55A868']
for i, metric in enumerate(metrics):
    axes[i].bar(results_df.index, results_df[metric], color=colors)
    axes[i].set_title(f'Model Comparison -- {metric}')
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

best = results_df['R2'].idxmax()
print(f'Best model: {best} (R2 = {results_df.loc[best, "R2"]})')